
# Joint Photometry + Spectroscopy Fit

Demonstrates tengri's Observation API for joint fitting across two data
streams. Creates a mock galaxy with SDSS photometry and low-resolution
spectroscopy, then recovers parameters via MAP. Shows how spectroscopy
breaks photometric degeneracies.

Reference: Conroy 2013 (ARA&A, 51, 393); Leja et al. 2019 on spectroscopic
constraints for star formation histories.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()

# No cache_dir: tengri resolves each curve across its own data directories, so
# an example works from any working directory. See plot_fisher_degeneracy.py
# for why the four-deep relative walk this replaces was a hazard.
phot = tengri.Photometry.from_names(["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"])
wave_rest = jnp.linspace(3800.0, 9200.0, 200)
z = 0.1
wave_obs = wave_rest * (1 + z)
spec_config = tengri.Spectroscopy(wave_obs=wave_obs)
obs = tengri.Observation(photometry=phot, spectroscopy=spec_config)

model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={"type": "tsnorm", "all_params": tengri.FREE, "skew": 0.3, "trunc": 5.0},
    dust={"type": "two_component", "all_params": tengri.FIXED},
    redshift=tengri.Fixed(0.1),
)

true_params = {
    "sfh_tsnorm_log_total_mass": 1.2,
    "sfh_tsnorm_peak_lbt_gyr": 1.5,
    "sfh_tsnorm_width_gyr": 2.0,
    "sfh_tsnorm_skew": 0.3,
    "sfh_tsnorm_trunc": 5.0,
    "met_logzsol": -0.3,
    "dust_tau_bc": 0.3,
    "dust_tau_diff": 0.4,
    "dust_slope": -0.7,
    "redshift": 0.1,
}

key = jax.random.PRNGKey(42)
flux_phot_true = model.predict_photometry(true_params)
flux_spec_true = model.predict_spectrum(true_params, wave_obs)
flux_true = jnp.concatenate([flux_phot_true, flux_spec_true])
noise = flux_true / 20.0
flux_obs = flux_true + noise * jax.random.normal(key, shape=flux_true.shape)

forward = tengri.ForwardModel.build(sed=model, observation=obs)
posterior = forward.fit(
    flux_obs,
    noise,
    method="map",
    optimizer="adam",
    n_steps=300,
    verbose=False,
)
fit_params = posterior.params

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
band_wave_um = np.array([3551, 4686, 6166, 7480, 8932]) / 1e4
phot_obs = np.array(flux_obs[:5])
phot_noise = np.array(noise[:5])
phot_fit = np.array(model.predict_photometry(fit_params))

ax.errorbar(band_wave_um, phot_obs, yerr=phot_noise, fmt="o", color="k", ms=6, label="Observed")
ax.plot(band_wave_um, phot_fit, "^", color="C3", ms=7, mfc="none", mew=1.5, label="MAP")
ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"Flux density [$\mu$Jy]")
ax.legend(frameon=False)

ax = axes[1]
wave_plot = np.array(wave_obs) / 1e4
n_phot = 5
spec_obs = np.array(flux_obs[n_phot:])
spec_fit = np.array(model.predict_spectrum(fit_params, wave_obs))
ax.plot(wave_plot, spec_obs, color="gray", lw=0.5, alpha=0.7, label="Observed")
ax.plot(wave_plot, spec_fit, color="C3", lw=1.5, label="MAP")
ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"Flux density [$\mu$Jy]")
ax.legend(frameon=False)

fig.tight_layout()
plt.savefig("plot_joint_fit.png", dpi=150, bbox_inches="tight")